# v3 — 5 classes, one per star
1★ strongly_negative … 5★ strongly_positive. Runtime → T4 GPU → Run all. ~1.5 h.
Expect lower accuracy than 3 classes (~0.65): judge it by off-by-one accuracy, MAE in stars and QWK.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Checkpoints go to Google Drive so a disconnect can be resumed. If the mount fails
# ("credential propagation was unsuccessful" happens with several Google accounts in one browser),
# fall back to the VM's disk: training still works, but a disconnect restarts from zero.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/btp_v3_star_level_5class'
except Exception as e:
    print(f'Drive mount failed ({e}); checkpoints stay on the VM.')
    DRIVE = '/content/btp_v3_star_level_5class'
!mkdir -p {DRIVE}
!test -d /content/BTP || git clone -q https://github.com/Abhijeet-SP/BTP.git /content/BTP
%cd /content/BTP
!git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
RESULTS = 'versions/v3_star_level_5class/results'
!pip -q install -U transformers datasets accelerate scikit-learn

In [ ]:
# ~8 min: 40k reviews per star -> data5/ (160k train, 20k val, 20k test) + 20k natural-mix test
!python prepare_data.py --classes 5 --per-class 40000

In [ ]:
!python train_baseline_tfidf.py --data-dir data5 --results-dir {RESULTS}

In [ ]:
# ~1 h on a T4
!python finetune_roberta.py --name roberta_base_5class --data-dir data5 --batch-size 32 --eval-steps 2500 --results-dir {RESULTS} --ckpt-dir {DRIVE}/ckpt

In [ ]:
import os
assert os.path.exists('models/roberta_base_5class/config.json'), 'Training failed: scroll up. Re-run the cell to resume from the checkpoint.'
print('Training finished OK')

In [ ]:
import json
m = json.load(open(f'{RESULTS}/metrics.json'))
for k in ['tfidf_5class', 'roberta_base_5class', 'roberta_base_5class_natural', 'roberta_base_5class_natural_prior']:
    if k in m:
        v = m[k]
        print(f"{k:28s} acc {v['accuracy']:.4f}  macro-F1 {v['macro_f1']:.4f}  "
              f"off-by-one {v['off_by_one_accuracy']:.4f}  MAE {v['mae_classes']:.3f}  QWK {v['quadratic_weighted_kappa']:.4f}")

In [ ]:
!zip -qr btp_v3_star_level_5class.zip models/roberta_base_5class versions/v3_star_level_5class/results && ls -lh btp_v3_star_level_5class.zip
!cp btp_v3_star_level_5class.zip {DRIVE}/
from google.colab import files
files.download('btp_v3_star_level_5class.zip')